# Phase 4 Step 2 — Kaggle DistilBERT fine-tuning

**Status: notebook prepared; remote run abhi verify nahi hui. Outputs intentionally empty hain.**

## Source strategy

| Component | Origin |
| --- | --- |
| **Project source code** | Cloned from GitHub at runtime: `https://github.com/Shubh6665/ai-phising-risk-platform.git`, branch `main` |
| **Phishing CSV datasets** | Attached Kaggle Input dataset (private; raw CSVs are never in the GitHub repo) |

Clean Kaggle session: Internet ON, GPU accelerator ON; keep datasets private.
No project snapshot dataset needs to be attached — Cell 1 clones the current repository directly.
The phishing CSV dataset must still be attached as a Kaggle Input.

Two manual approvals are required before proceeding: truncation review before training, validation review before upload.
Run All stops at each approval gate by design.
Do not assume GPU/runtime/quota results in advance.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Torch import se pehle single GPU visibility select karo; no distributed experiment.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ---------------------------------------------------------------------------
# PROJECT SOURCE: GitHub clone
# ---------------------------------------------------------------------------
# Project source code GitHub se clone hota hai at runtime.
# Phishing CSV datasets /kaggle/input se aate hain (attached Kaggle dataset).
# Dono ko kabhi mix nahi karte: raw CSVs git mein nahi hain.
# ---------------------------------------------------------------------------

GITHUB_REPO = 'https://github.com/Shubh6665/ai-phising-risk-platform.git'
GITHUB_BRANCH = 'main'
PROJECT_ROOT = Path('/kaggle/working/AI-PhishingRiskPlatform')

# Safety check: existing copy overwrite nahi hogi (clean session required).
if PROJECT_ROOT.exists():
    raise FileExistsError(
        'Clean session required; ' + str(PROJECT_ROOT) + ' already exists. '
        'New Kaggle session start karo ya manually remove karo.'
    )

# --- Step 1: Clone from GitHub -------------------------------------------
print('Cloning ' + GITHUB_REPO + ' (branch: ' + GITHUB_BRANCH + ') ...')
clone_result = subprocess.run(
    [
        'git', 'clone',
        '--branch', GITHUB_BRANCH,
        '--single-branch',
        '--depth', '1',
        GITHUB_REPO,
        str(PROJECT_ROOT),
    ],
    capture_output=True,
    text=True,
)
if clone_result.returncode != 0:
    raise RuntimeError(
        'GitHub clone failed (exit ' + str(clone_result.returncode) + ').'
        ' stdout: ' + clone_result.stdout
        + ' stderr: ' + clone_result.stderr
        + ' Kaggle Internet switch ON hai? Repository public/accessible hai?'
    )
print('Clone complete:', PROJECT_ROOT)

# --- Step 2: Verify expected project files are present -------------------
REQUIRED_FILES = [
    'pyproject.toml',
    'src/models/nlp_classifier.py',
    'docs/distilbert_data_manifest.json',
]
missing = [f for f in REQUIRED_FILES if not (PROJECT_ROOT / f).is_file()]
if missing:
    raise RuntimeError(
        'Cloned repository mein required files missing hain: '
        + str(missing)
        + ' Ye files GitHub pe push hui hain? Branch sahi hai?'
    )
print('Required files verified:', REQUIRED_FILES)

# --- Step 3: Capture and print the cloned commit for traceability --------
rev_result = subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
    capture_output=True,
    text=True,
)
if rev_result.returncode != 0:
    raise RuntimeError(
        'Cloned repository ka commit SHA verify nahi ho saka.'
        ' stderr: ' + rev_result.stderr
    )
CLONED_COMMIT = rev_result.stdout.strip()
if not CLONED_COMMIT or len(CLONED_COMMIT) != 40:
    raise RuntimeError(
        'Unexpected commit SHA format: ' + repr(CLONED_COMMIT) + '. '
        'Repository state unexpected hai.'
    )
print('Repository:', GITHUB_REPO)
print('Branch:    ', GITHUB_BRANCH)
print('Commit:    ', CLONED_COMMIT)

# --- Step 4: Install the cloned project in editable mode -----------------
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT) + '[dev,data,nlp]'],
    check=False,
)
if install_result.returncode != 0:
    raise RuntimeError(
        'pip install of cloned project failed. '
        'pyproject.toml correct hai? Dependencies available hain?'
    )
sys.path.insert(0, str(PROJECT_ROOT))
print('Installed source:', PROJECT_ROOT)
print('Dataset files remain in /kaggle/input -- not cloned from GitHub.')

## GPU aur reproducibility
GPU name/memory actual runtime se print honge; T4 assume nahi. Package resolver environment choose karta hai, so resolved versions aur `pip freeze` run report ke saath save karte hain. CPU fallback forbidden. Seed 42 reproducibility improve karta hai, but different GPU/kernel/library versions par bitwise training identity guarantee nahi.

In [ ]:
import importlib.metadata
import json
import platform
import torch
from huggingface_hub import HfApi
from src.data.loader import SOURCE_FILES
from src.data.nlp_dataset import load_nlp_development_data, verify_data_manifest
from src.models.nlp_classifier import (BASE_CHECKPOINT, NLPTrainingConfig, training_token_lengths, choose_max_length, create_distilbert_components, tokenize_development, build_trainer, transformer_metrics, sample_probabilities, verify_probability_reload)

if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError('Clean session mein exactly one visible CUDA GPU required')
GPU = {'name': torch.cuda.get_device_name(0), 'total_memory_bytes': torch.cuda.get_device_properties(0).total_memory, 'visible_devices': torch.cuda.device_count()}
VERSIONS = {'python': platform.python_version(), 'cuda': torch.version.cuda}
VERSIONS.update({name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'datasets', 'accelerate', 'huggingface-hub', 'numpy', 'pandas', 'scikit-learn']})
print(json.dumps({'gpu': GPU, 'versions': VERSIONS}, indent=2))

In [ ]:
raw_dirs = [p.parent for p in Path('/kaggle/input').rglob(SOURCE_FILES['CEAS_08'])
            if all((p.parent / filename).is_file() for filename in SOURCE_FILES.values())]
if len(raw_dirs) != 1:
    raise RuntimeError(f'Six original source CSVs ka exactly one directory required: {raw_dirs}')
data = load_nlp_development_data(raw_dirs[0])
verify_data_manifest(data.manifest, PROJECT_ROOT / 'docs/distilbert_data_manifest.json')
print(json.dumps(data.manifest, indent=2))
assert data.manifest['counts'] == {'train': 57574, 'validation': 12337, 'test': 12338}
# Adapter test text/labels training arguments mein expose nahi karta.
print('Model inputs: cleaned subject + two newlines + body; source/label text features nahi.')

## Tokenizer aur first-token classification
`AutoTokenizer` pretrained uncased WordPiece vocabulary reuse karta hai; no tokenizer fitting. Subwords words/URLs ko multiple pieces bana sakte hain. `input_ids` vocabulary indices; `attention_mask` real tokens=1, padding=0. `[CLS]` first position ka final contextual representation classification head ko jata hai. DistilBERT head pre-classifier/ReLU/dropout/classifier se two logits banata hai. Head new hai; encoder aur head dono train honge. Neeche demonstration synthetic hai, real raw email Hub par publish nahi karna.

In [ ]:
BASE_REVISION = HfApi().model_info(BASE_CHECKPOINT).sha
tokenizer, model = create_distilbert_components(BASE_REVISION)
print('Base checkpoint/revision:', BASE_CHECKPOINT, BASE_REVISION)
print('Tokenizer:', type(tokenizer).__name__, 'vocabulary:', tokenizer.vocab_size, 'model limit:', tokenizer.model_max_length)
example = 'URGENT account verification\n\nPlease verify your account at https://example.com/login'
encoded_example = tokenizer(example, add_special_tokens=True, return_token_type_ids=False)
print('Tokens:', tokenizer.convert_ids_to_tokens(encoded_example['input_ids']))
print('IDs:', encoded_example['input_ids'])
print('Attention mask:', encoded_example['attention_mask'])
print('Label mapping:', model.config.id2label)
assert model.config.num_labels == 2 and tokenizer.is_fast
assert model.config.max_position_embeddings >= 256

## Length choice: distribution first, truncation visible
Full training-only token lengths measure karte hain, including `[CLS]`/`[SEP]`, without truncation. Encoder ko long inputs feed nahi hote. Initial memory budget cap 256, coverage target 90%; candidates 128/256 mein smallest meeting target choose hota hai. If neither meets target, cap 256 selected with unmet-target warning and measured truncated fraction. Yeh predeclared rule hai—not validation-metric tuning.

Right truncation subject-first prefix preserve karti hai, but tail URLs/quoted context/closing instructions lose ho sakte hain. Dynamic padding per batch longest sequence tak hogi. Distribution/output review kiye bina training approve mat karo. OOM par automatic fallback nahi: batch 8→4 and accumulation 2→4 keeps effective batch 16; lower length more information discard karegi. Any retry/new config report mein explicitly record karo.

In [ ]:
lengths = training_token_lengths(tokenizer, data.train_text)
length_decision = choose_max_length(lengths, memory_cap=256, target_coverage=0.90)
print(json.dumps(length_decision, indent=2))
if not length_decision['target_met']:
    print('WARNING: 90% coverage unmet; memory cap ke kaaran many emails truncate honge.')
config = NLPTrainingConfig(max_length=length_decision['max_length'])
# Saved tokenizer ka default limit bhi training choice preserve karega.
tokenizer.model_max_length = config.max_length
print('Proposed run:', json.dumps(config.to_dict(), indent=2))
del lengths

In [ ]:
APPROVE_TRUNCATION_AND_TRAINING = False
if not APPROVE_TRUNCATION_AND_TRAINING:
    raise RuntimeError('Actual length report review karo; phir is flag ko True karke continue karo')
RUN_ROOT = Path('/kaggle/working/model_artifacts/nlp')
OUTPUT_DIR = RUN_ROOT / 'distilbert-step2-checkpoints'
EXPORT_DIR = RUN_ROOT / 'distilbert-step2-export'
REPORT_PATH = RUN_ROOT / 'distilbert-step2-report.json'
if OUTPUT_DIR.exists() or EXPORT_DIR.exists() or REPORT_PATH.exists():
    raise FileExistsError('Existing experiment overwrite nahi hoga; investigate/restart explicitly')
development = tokenize_development(tokenizer, data.train_text, data.train_labels, data.validation_text, data.validation_labels, max_length=config.max_length)
trainer = build_trainer(model, tokenizer, development, config, OUTPUT_DIR)
preview = trainer.data_collator([development['train'][i] for i in range(3)])
print('Batch shapes:', {name: list(tensor.shape) for name, tensor in preview.items()})
print('Trainer device:', trainer.args.device, 'model device:', next(trainer.model.parameters()).device)
assert trainer.args.device.type == 'cuda' and next(trainer.model.parameters()).is_cuda
assert set(development) == {'train', 'validation'}

## Initial fine-tuning configuration
3 epochs; LR 2e-5; weight decay 0.01; one GPU batch 8 × accumulation 2 = effective batch 16. AdamW, linear scheduler, 10% warmup; seed/data seed 42, FP16 mixed precision. Encoder/head both update; tokenizer fixed. Evaluate/save each epoch; restore highest-validation-F1 checkpoint, keep at most two checkpoints. No sweep, no class weighting, no threshold tuning, no test. Step-wise logged train losses and epoch validation losses compare cautiously: train uses dropout, validation eval mode; they are not identical measurement protocols.

Network/model download, GPU quota/runtime and OOM failures actual errors hain—hide mat karo. Failure report export hoga; configuration silently alter nahi hogi.

In [ ]:
try:
    training_result = trainer.train()
except Exception as error:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    REPORT_PATH.with_name('distilbert-step2-failure.json').write_text(json.dumps({'status': 'training_failed', 'error_type': type(error).__name__, 'error': str(error), 'configuration': config.to_dict(), 'gpu': GPU, 'versions': VERSIONS, 'training_history': trainer.state.log_history}, indent=2))
    raise
validation_prediction = trainer.predict(development['validation'], metric_key_prefix='validation')
validation_metrics = transformer_metrics(validation_prediction)
print(json.dumps({'epochs_completed': trainer.state.epoch, 'training_metrics': training_result.metrics, 'validation_metrics': validation_metrics, 'validation_loss': validation_prediction.metrics['validation_loss'], 'best_checkpoint': trainer.state.best_model_checkpoint, 'history': trainer.state.log_history}, indent=2))

In [ ]:
classical = json.loads((PROJECT_ROOT / 'docs/classical_ml_development.json').read_text())['validation_metrics']
tfidf = json.loads((PROJECT_ROOT / 'docs/tfidf_validation_reference.json').read_text())['metrics']
comparison = {'handcrafted_lr': classical['lr_unweighted'], 'handcrafted_rf': classical['rf_unweighted'], 'tfidf_lr': tfidf, 'distilbert': validation_metrics}
print(json.dumps(comparison, indent=2))
print('Historical validation baselines reuse hue; final saved RF aur held-out test use nahi hue.')
print('Source/style/template shortcuts sab models ke liye limitation hain; high score phishing understanding ka proof nahi.')

In [ ]:
import hashlib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sample = data.validation_text[:32]  # fixed validation prefix; no new random split.
reference_probabilities = sample_probabilities(trainer.model, tokenizer, sample, max_length=config.max_length)
trainer.save_model(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))
reloaded_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR, use_fast=True)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(EXPORT_DIR, use_safetensors=True).to('cuda')
assert reloaded_tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_tokenizer.model_max_length == config.max_length
assert reloaded_model.config.id2label == trainer.model.config.id2label
reloaded_probabilities = sample_probabilities(reloaded_model, reloaded_tokenizer, sample, max_length=config.max_length)
local_reload = verify_probability_reload(reference_probabilities, reloaded_probabilities)
source_hashes = {str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted((PROJECT_ROOT / 'src').rglob('*.py'))}
report = {'status': 'validation_complete_hub_pending', 'base_checkpoint': BASE_CHECKPOINT, 'base_revision': BASE_REVISION, 'configuration': config.to_dict(), 'tokenizer': {'class': type(tokenizer).__name__, 'vocab_size': tokenizer.vocab_size, 'do_lower_case': tokenizer.do_lower_case, 'truncation_side': tokenizer.truncation_side, 'padding_side': tokenizer.padding_side, 'max_length': config.max_length}, 'length_decision': length_decision, 'data_manifest': data.manifest, 'project_source_sha256': source_hashes, 'gpu': GPU, 'versions': VERSIONS, 'epochs_completed': trainer.state.epoch, 'training_metrics': training_result.metrics, 'validation_metrics': validation_metrics, 'validation_loss': validation_prediction.metrics['validation_loss'], 'training_history': trainer.state.log_history, 'best_checkpoint': trainer.state.best_model_checkpoint, 'comparison': comparison, 'local_reload': local_reload, 'test_evaluated': False, 'hf_model_id': None}
REPORT_PATH.write_text(json.dumps(report, indent=2))
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=(RUN_ROOT / 'environment-requirements.txt').open('w'))
print('Local reload:', local_reload)
print('Measured report:', REPORT_PATH)
# Training optimizer/model GPU storage release before Hub reload.
trainer.model.to('cpu')
reloaded_model.to('cpu')
del trainer, reloaded_model
torch.cuda.empty_cache()

## HF publication: manual approval, no secrets in notebook
Validation loss/history + FP/FN + TF-IDF comparison review karo; satisfactory run ke baad upload approve karo. Public training dataset ka matlab unrestricted licensing/privacy nahi. Weights memorization risk rakhte hain. Default private HF repository; public only after explicit licensing/privacy review. Kaggle Secrets mein `HF_TOKEN` (write-scoped) add/enable karo. Token print/cell literal/login cache mein store mat karo. Raw emails, optimizer checkpoints aur secrets upload nahi honge; export model/tokenizer + aggregate model card only.

HF network calls quota/storage policies follow karte hain. Upload/reload failures par Hub complete claim nahi karna. HF model identifier account owner se runtime par derive hoga—abhi actual ID unavailable hai.

In [ ]:
APPROVE_VALIDATION_FOR_UPLOAD = False
PUBLIC_HUB_REPOSITORY = False
PUBLIC_LICENSE_PRIVACY_REVIEWED = False
HF_REPO_NAME = 'phishing-risk-distilbert'
if not APPROVE_VALIDATION_FOR_UPLOAD:
    raise RuntimeError('Validation/loss/reload report review karke upload explicitly approve karo')
if PUBLIC_HUB_REPOSITORY and not PUBLIC_LICENSE_PRIVACY_REVIEWED:
    raise RuntimeError('Public upload se pehle dataset license/privacy review required')
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret('HF_TOKEN')
api = HfApi(token=hf_token)
repo_id = api.whoami()['name'] + '/' + HF_REPO_NAME
api.create_repo(repo_id, private=not PUBLIC_HUB_REPOSITORY, exist_ok=False)
card = '---\nlanguage: en\n\npipeline_tag: text-classification\nlibrary_name: transformers\n---\n# AI Phishing DistilBERT\n\nBinary target: legitimate vs phishing/spam. Source/template shortcuts aur historical mixed labels limit generalization; production security guarantee nahi. Dataset licensing/publication rights independently review karo.\n\nBase: ' + BASE_CHECKPOINT + '@' + BASE_REVISION + '\n\nConfiguration:\n```json\n' + json.dumps(config.to_dict(), indent=2) + '\n```\nValidation metrics:\n```json\n' + json.dumps(validation_metrics, indent=2) + '\n```\nData/split manifest:\n```json\n' + json.dumps(data.manifest, indent=2) + '\n```\nNo test evaluation. No raw email examples published.\n'
card += '\n\nLength/truncation report:\n```json\n' + json.dumps(length_decision, indent=2) + '\n```\nRuntime/GPU:\n```json\n' + json.dumps({'gpu': GPU, 'versions': VERSIONS, 'epochs_completed': report['epochs_completed'], 'training_metrics': report['training_metrics'], 'validation_loss': report['validation_loss']}, indent=2) + '\n```\n'
(EXPORT_DIR / 'README.md').write_text(card)
api.upload_folder(repo_id=repo_id, folder_path=str(EXPORT_DIR), commit_message='Upload validation-selected DistilBERT model and tokenizer')
hub_revision = api.model_info(repo_id).sha
hub_tokenizer = AutoTokenizer.from_pretrained(repo_id, revision=hub_revision, token=hf_token, use_fast=True, cache_dir=RUN_ROOT / 'hub-reload-cache')
hub_model = AutoModelForSequenceClassification.from_pretrained(repo_id, revision=hub_revision, token=hf_token, use_safetensors=True, cache_dir=RUN_ROOT / 'hub-reload-cache').to('cuda')
assert hub_tokenizer.get_vocab() == tokenizer.get_vocab()
assert hub_tokenizer.model_max_length == config.max_length
assert hub_model.config.id2label == {0: 'legitimate', 1: 'phishing_spam'}
hub_probabilities = sample_probabilities(hub_model, hub_tokenizer, sample, max_length=config.max_length)
hub_reload = verify_probability_reload(reference_probabilities, hub_probabilities)
report.update(status='validation_and_hub_reload_complete', hf_model_id=repo_id, hf_revision=hub_revision, hf_private=not PUBLIC_HUB_REPOSITORY, hub_reload=hub_reload)
REPORT_PATH.write_text(json.dumps(report, indent=2))
print(json.dumps({'hf_model_id': repo_id, 'hf_revision': hub_revision, 'reload': hub_reload, 'report': str(REPORT_PATH)}, indent=2))
del hf_token

## Final handoff aur stop
Download `distilbert-step2-report.json` (plus failure report if any) aur notebook executed outputs. GPU name, selected max length/truncation fraction, config, actual epochs/losses, validation metrics/confusion, comparison, local/Hub reload aur exact HF ID isi report se repository docs update honge. **Remote results aane se pehle documentation mein metrics invent mat karo.** Weights git mein nahi; committed notebook ke outputs clear karo and secret scan karo.

Training-vs-validation loss behavior inspect karo: rising validation loss with declining train loss overfitting ka signal ho sakta hai; best F1 checkpoint and last-epoch loss different ho sakte hain. One split/source shortcuts real-world phishing understanding prove nahi karte.

Yahin stop. No test evaluation, FastAPI, RAG, LangGraph or later-phase work.